In [1]:

import os 
import sys
from dataclasses import dataclass
sys.path.append(os.path.join(os.path.dirname("__file__"), '..'))

import numpy as np
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix, diags
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

import skfem
import gmsh


from src_copy.utils import *
from src_copy.core import *
from src_copy.solver import *

In [4]:
def run_circle_obs_fig(config: Config, save_result=False, num_particle=30, n_snapshot=6):
    
    logger = Experiment(config.name_path)

    bound=(0, 2, 0, 1)
    circles= [((.9, .5), .4)]
    rounded_rects=[]
    nodes0, eles0 = build_dom_example(
        bound, circles, rounded_rects,
        h_min=config.h_min, h_max=config.h_max
    )

    dom = DOMobs1(3, circles, rounded_rects)
    mesh = Mesh(nodes0, eles0, dom)
    E0 = Ekern(nodes0, gamma=config.gamma, sigma=config.sigma)

    def _make_marg(nodes, mass):
        x, y = nodes.T
        
        g1 = np.exp(-((x - 0.35) ** 2 + (y - 0.30) ** 2) / (2.0 * 0.08 ** 2))
        g2 = np.exp(-((x - 0.35) ** 2 + (y - 0.70) ** 2) / (2.0 * 0.08 ** 2))
        mu0= 0.5 * (g1 + g2) + 1e-12
        
        dx = np.maximum(0., np.maximum(1.5-x, x-1.8))
        dy = np.maximum(0., np.maximum(.35-y, y-.85))
        muT = np.exp(-(dx**2 + dy**2) / (2.*.05**2)) + 1e-12
        
        return mu0 / np.sum(mass*mu0), muT / np.sum(mass*muT)
    
    
    solution = solve_mfrsb(mesh, E0, SLStenc0, config, _make_marg, logger) 
    
    if save_result:
        box = [(1.5, 1.8, 0.35, 0.85)]
        trajs, max_penetration = path_sim(mesh, solution, num_particle, 3, config.seed)
        save_snapshot2d(mesh, solution, 
                        os.path.join(config.name_path, "snapshot.png"),
                        trajs, box, n_snapshot, "MFRSB with Circular Obs")
        logger.save_sol("solution_circle", mesh, solution, trajs, box)
        
    return solution, logger

In [5]:
config = Config()
run_circle_obs_fig(config, save_result=True, num_particle=200, n_snapshot=10)

  [Iter 01] ||drho||_inf: 1.48e+01 | q:    --- | L1(0,T): (5.61e-01, 5.24e-01) | W2: 4.31e-01 | MassErr: 1.1e-15 | MaxPen: 0.0e+00 | CFL: 0.01 | dt_CPU: 1.83s
  [Iter 02] ||drho||_inf: 1.47e+01 | q: 0.991 | L1(0,T): (1.96e-01, 1.83e-01) | W2: 2.33e-01 | MassErr: 8.9e-16 | MaxPen: 0.0e+00 | CFL: 0.02 | dt_CPU: 1.83s
  [Iter 03] ||drho||_inf: 4.54e+00 | q: 0.309 | L1(0,T): (6.87e-02, 6.42e-02) | W2: 1.20e-01 | MassErr: 1.7e-15 | MaxPen: 0.0e+00 | CFL: 0.02 | dt_CPU: 1.82s
  [Iter 04] ||drho||_inf: 1.59e+00 | q: 0.350 | L1(0,T): (2.40e-02, 2.25e-02) | W2: 5.37e-02 | MassErr: 1.1e-15 | MaxPen: 0.0e+00 | CFL: 0.02 | dt_CPU: 1.81s
  [Iter 05] ||drho||_inf: 6.13e-01 | q: 0.385 | L1(0,T): (8.42e-03, 7.86e-03) | W2: 3.88e-03 | MassErr: 8.9e-16 | MaxPen: 0.0e+00 | CFL: 0.02 | dt_CPU: 1.82s
  [Iter 06] ||drho||_inf: 2.66e-01 | q: 0.434 | L1(0,T): (2.95e-03, 2.75e-03) | W2: 2.30e-03 | MassErr: 1.1e-15 | MaxPen: 0.0e+00 | CFL: 0.03 | dt_CPU: 1.81s
  [Iter 07] ||drho||_inf: 1.27e-01 | q: 0.478 | L1(

({'rho': array([[4.68702107e-11, 7.73717627e-07, 4.68702107e-11, ...,
          4.68702107e-11, 4.68702107e-11, 4.68702107e-11],
         [4.95343766e-11, 2.20523719e-05, 5.19448539e-11, ...,
          4.66499951e-11, 4.76712998e-11, 4.51297988e-11],
         [5.13809813e-11, 3.07719993e-04, 5.61922873e-11, ...,
          4.65287605e-11, 4.82855533e-11, 4.38454365e-11],
         ...,
         [1.20075424e+00, 1.41832547e-09, 1.27675467e-05, ...,
          7.73185636e-02, 5.41369614e-01, 4.68916897e-01],
         [5.99002632e-01, 1.44999097e-09, 3.25176616e-08, ...,
          6.44549978e-02, 5.60270231e-01, 1.44567312e-01],
         [1.26203825e-03, 2.55392603e-11, 2.55681577e-11, ...,
          7.15294623e-03, 4.46836664e-01, 2.25168610e-04]], shape=(41, 1220)),
  'f': array([[ 10.11255108,  -5.20613298,   9.94887996, ...,  11.11541051,
           11.00521802,   9.83252758],
         [ 10.10444814,  -5.51994744,   9.92853965, ...,  11.12744677,
           11.00828474,   9.81229266],
  